# R&D 특구 유사 과제 검색 프로토타입

준비 중인 연구 주제를 입력하면, R&D 특구 소속 기관이 수행한 유사 과제와 그 과제를 수행한 연구자를
찾아줍니다.

In [1]:
import pandas as pd

DATA_DIR = "../01_data"

companies = pd.read_csv(f"{DATA_DIR}/특구입주기업현황.csv", encoding="cp949")
tasks_raw = pd.read_csv(f"{DATA_DIR}/이알앤디_과제정보.csv", encoding="cp949")

print("특구입주기업현황:", len(companies), "건")
print("이알앤디_과제정보 (원본):", len(tasks_raw), "건")

tasks = tasks_raw[tasks_raw["보안과제여부"] != "Y"].copy()
print("보안과제 제외 후:", len(tasks), "건")
assert len(tasks) == 11783, f"보안과제 제외 후 건수가 예상과 다름: {len(tasks)}"

특구입주기업현황: 6359 건
이알앤디_과제정보 (원본): 11788 건
보안과제 제외 후: 11783 건


In [2]:
companies = companies.rename(columns={"지역": "특구지역"})
companies["_key"] = companies["기관명"].str.strip()
tasks["_key"] = tasks["주관기관명"].str.strip()

companies_dedup = companies.drop_duplicates(subset="_key")

df = (
    tasks.merge(companies_dedup[["_key", "특구지역"]], on="_key", how="inner")
    .drop(columns=["_key"])
    .reset_index(drop=True)
)

print("조인 결과:", len(df), "건 /", df["주관기관명"].nunique(), "개 기관")
assert len(df) == 1680, f"조인 건수가 예상과 다름: {len(df)}"
assert df["주관기관명"].nunique() == 27, f"조인 기관 수가 예상과 다름: {df['주관기관명'].nunique()}"
assert df["연구자번호"].nunique() == 1464, f"연구자 수가 예상과 다름: {df['연구자번호'].nunique()}"

조인 결과: 1680 건 / 27 개 기관


In [3]:
_name_org_ids = (
    df.drop_duplicates(subset=["연구책임자명", "주관기관명", "연구자번호"])
    .groupby(["연구책임자명", "주관기관명"])["연구자번호"]
    .nunique()
)
DUP_NAME_ORG_PAIRS = set(_name_org_ids[_name_org_ids > 1].index)
print("동명이인 표기 대상 (이름+기관) 쌍:", len(DUP_NAME_ORG_PAIRS))
assert len(DUP_NAME_ORG_PAIRS) == 11, f"동명이인 쌍 수가 예상과 다름: {len(DUP_NAME_ORG_PAIRS)}"

def display_researcher_name(row, dup_pairs=DUP_NAME_ORG_PAIRS):
    key = (row["연구책임자명"], row["주관기관명"])
    if key in dup_pairs:
        return f"{row['연구책임자명']}({str(int(row['연구자번호']))[-4:]})"
    return row["연구책임자명"]

동명이인 표기 대상 (이름+기관) 쌍: 11


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df["과제명"])
print("TF-IDF 행렬 크기:", tfidf_matrix.shape)

svd = TruncatedSVD(n_components=2, random_state=42)
coords = svd.fit_transform(tfidf_matrix)

N_CLUSTERS = 6  # Task 9에서 화면2 산점도를 보고 최종 확정

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
clusters = kmeans.fit_predict(coords)

df["_cluster"] = clusters
print("클러스터별 건수:")
print(df["_cluster"].value_counts().sort_index())

TF-IDF 행렬 크기: (1680, 8037)


클러스터별 건수:
_cluster
0    253
1    463
2     84
3    584
4    160
5    136
Name: count, dtype: int64


In [5]:
sizes = df["_cluster"].value_counts()
assert len(sizes) == N_CLUSTERS, f"빈 클러스터 존재: {len(sizes)}/{N_CLUSTERS}"
assert (sizes > 0).all()